# 04 — Dynamic SHAP analysis
Compute SHAP values per out-of-sample year for XGBoost and visualise how feature importance evolves.

In [ ]:
import sys, warnings, pickle
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

with open('../results/walk_forward_results.pkl', 'rb') as f:
    saved = pickle.load(f)

results       = saved['results']
feature_names = saved['feature_names']
print("Features:", feature_names)


## Compute dynamic SHAP

> ⏱ SHAP computation takes ~1–2 minutes.

In [ ]:
from src.evaluation import compute_dynamic_shap

xgb_inputs = results.get('XGBoost', {}).get('shap_inputs', [])
shap_df = compute_dynamic_shap(xgb_inputs, feature_names)

print(f"SHAP computed for {len(shap_df)} years")
shap_df.round(3)


## Heatmap: feature importance by year

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(shap_df.T.values, aspect='auto', cmap='YlOrRd',
               vmin=0, vmax=shap_df.values.max())

ax.set_xticks(range(len(shap_df.index)))
ax.set_xticklabels(shap_df.index.astype(int), rotation=45, ha='right')
ax.set_yticks(range(len(shap_df.columns)))
ax.set_yticklabels(shap_df.columns)
ax.set_title('Dynamic feature importance — XGBoost SHAP values (normalised)', fontsize=12)
plt.colorbar(im, ax=ax, label='Mean |SHAP| (relative)')
plt.tight_layout()
plt.savefig('../results/figures/shap_heatmap.png', bbox_inches='tight')
plt.show()


## Stacked area: importance over time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#378ADD','#1D9E75','#BA7517','#D85A30','#D4537E','#7F77DD','#888780','#5DCAA5','#F0997B']

bottom = np.zeros(len(shap_df))
for i, col in enumerate(shap_df.columns):
    vals = shap_df[col].values * 100
    ax.fill_between(shap_df.index.astype(int), bottom, bottom + vals,
                    alpha=0.85, color=colors[i % len(colors)], label=col)
    bottom += vals

ax.set_ylim(0, 100)
ax.set_ylabel('Feature importance (%)')
ax.set_title('Stacked feature importance over time (XGBoost SHAP)', fontsize=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('../results/figures/shap_stacked.png', bbox_inches='tight')
plt.show()


## Key findings

Interpret what the model learns:

In [ ]:
print("Top feature by year:")
print(shap_df.idxmax(axis=1).to_string())

print("\nMean importance across full sample:")
print(shap_df.mean().sort_values(ascending=False).round(3).to_string())

print("\nChange in CMA importance (2009 → last year):")
cma_trend = shap_df['beta_CMA'].iloc[-1] - shap_df['beta_CMA'].iloc[0]
print(f"  {cma_trend:+.3f} ({cma_trend/shap_df['beta_CMA'].iloc[0]*100:+.1f}%)")


## Save SHAP results

In [ ]:
shap_df.to_csv('../results/shap_dynamic.csv')
print("Saved to results/shap_dynamic.csv")
